In [19]:
# =============================================================================
# doa_montecarlo.jl (v2) -- Monte Carlo error-source analysis built around YOUR
# three scripts: MUSIC, Trig+FFT, LUT+cost fn.
#
# Experiments: A noise sensitivity   B mic geometry   C channel mismatch
#              D frame-to-frame variability
#
# Usage:   include("doa_montecarlo.jl");  run_experiments(ntrials = 20)  # time it first
#          run_experiments()                                             # full run
# Packages: ] add FFTW DataFrames CSV WAV Plots XLSX
# Outputs (in ./results): raw trial CSVs, summary CSVs, error_budget.csv, clean_estimates.csv,
#                         summary.xlsx (all summary tables, one sheet each), PNG plots
#
# Geometry/conventions are exactly yours: theta CCW from +x,
#   tau21 = (p1-p2).u/c , tau31 = (p1-p3).u/c ,
#   steering = [1, exp(-j2*pi*f*tau21), exp(-j2*pi*f*tau31)].
# With MICS_NOMINAL below, (p1-p2) = b(1/2,-sqrt3/2) and (p1-p3) = (b,0), which
# reproduces your sin(30deg - theta) and cos(theta) expressions exactly.
# =============================================================================

using FFTW, LinearAlgebra, Statistics, Random, DataFrames, CSV, WAV, Plots, XLSX

# ----------------------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------------------
const C_SOUND  = 343.0
const BASELINE = 0.017                       # side of the equilateral triangle (m)
const F_MIN, F_MAX, F_CUTOFF = 2000.0, 10088.0, 2000.0
const FRAME_LEN = 1024
const HOP       = FRAME_LEN ÷ 2

# rows = mic 1,2,3 ; columns = x,y  (same positions as your MUSIC / Trig scripts)
const MICS_NOMINAL = [ BASELINE/2             -BASELINE*sqrt(3)/6
                       0.0                     BASELINE*sqrt(3)/3
                      -BASELINE/2             -BASELINE*sqrt(3)/6 ]

const REC_DIR = "C:/Users/Morena Mofokeng/Desktop/Research project/recordings/4kHz-Sine"
const RECORDINGS = Dict(                     # filename => true azimuth (deg); NaN if unknown
    "180cm_0_d_man_sin_4000hz.wav" => 0.0,   # truth read from filename -- check
    "180cm_22.5_d_man_sin_4000hz.wav" => 22.5,
    "180cm_45_d_man_sin_4000hz.wav" => 45.0,
    "180cm_90_d_man_sin_4000hz.wav" => 90.0,
    "180cm_180_d_man_sin_4000hz.wav" => 180.0,
    "180cm_270_d_man_sin_4000hz.wav" => 270.0
)

# Oversampling factor per algorithm, as in your scripts (MUSIC 1, Trig 1, LUT 2).
# Set all to 1 to compare the algorithms on an identical front-end.
const OSF = (music = 1, trig = 1, lut = 2)

# LUT script uses ONE tone frequency (first frame's strongest bin) for every frame.
# :first_frame = your script;  :per_frame = use each frame's own bin frequency.
const LUT_FREQ_MODE  = :first_frame
const LUT_STEP_DEG   = 0.1                   # your script: 0.01 (10x slower, <0.03 deg difference)
const MUSIC_STEP_DEG = 0.1                   # as in your script

const SEED             = 1234
const N_TRIALS         = 100
const ANALYSIS_SECONDS = 3.0                 # loudest window of each file used for A/B/C (nothing = whole file)
const OUTLIER_DEG      = 15.0

# Experiment levels
const SNR_DB           = [40.0, 30.0, 20.0, 10.0, 5.0, 0.0, -5.0]   # in-band (F_MIN..F_MAX) SNR of ADDED noise
const POS_SIGMA_MM     = [0.25, 0.5, 1.0, 2.0]                      # per coordinate (baseline is 17 mm!)
const GAIN_SIGMA_DB    = [0.25, 0.5, 1.0, 2.0]
const DELAY_SIGMA_US   = [0.5, 1.0, 2.0, 5.0, 10.0]                 # max physical delay is b/c = 49.6 us
const BLOCK_LENS       = [4096, 8192, 16384, 32768]                 # D2: observation length (samples)
const N_BLOCKS         = 200
const VAD_DB           = 20.0                # D2: only blocks within this many dB of the loudest

const TYPICAL = Dict("A" => 10.0, "B" => 0.5, "C-gain" => 1.0, "C-delay" => 2.0, "D-block" => 8192.0)

# ----------------------------------------------------------------------------
# 2. Helpers
# ----------------------------------------------------------------------------
wrap180(a) = mod(a + 180.0, 360.0) - 180.0
wrappi(a)  = mod(a + π, 2π) - π
circmean_deg(a) = mod(atand(mean(sind.(a)), mean(cosd.(a))), 360.0)
unit(th_deg) = vcat(cosd.(collect(th_deg))', sind.(collect(th_deg))')       # 2 x nθ
const WIN = 0.5 .- 0.5 .* cos.(2π .* (0:FRAME_LEN-1) ./ FRAME_LEN)

# tau_1k(theta) = (p1 - pk).u / c   for k = 2, 3   (vectors over theta)
tau1k(mics, k, U) = vec((mics[1, :] .- mics[k, :])' * U) ./ C_SOUND

# your ifft_zeropad, unchanged
function ifft_zeropad(X::AbstractVector, M::Integer)
    N = length(X); @assert M ≥ N
    Xp = zeros(eltype(X), M); h = N ÷ 2
    Xp[1:h] .= X[1:h]
    Xp[end-(N-h)+1:end] .= X[h+1:end]
    if iseven(N) && M > N
        Xp[h+1] = X[h+1] / 2
        Xp[end-h+1] = X[h+1] / 2
    end
    return ifft(Xp) .* (M / N)
end

"High-pass at F_CUTOFF (symmetric mask) + oversample. Returns (y, fs_os)."
function highpass_osf(x, fs, osf)
    N = size(x, 1)
    X = fft(x, 1)
    sc = Int(round(F_CUTOFF / (fs / N)))
    H = zeros(N); H[sc+1:N-sc+1] .= 1.0
    X .*= H
    y = hcat([real(ifft_zeropad(X[:, m], N * osf)) for m in 1:3]...)
    return y, fs * osf
end

function stft3(y)
    N = size(y, 1); nf = 1 + div(N - FRAME_LEN, HOP); nb = FRAME_LEN ÷ 2 + 1
    S = Array{ComplexF64}(undef, nb, nf, 3)
    for m in 1:3, i in 1:nf
        S[:, i, m] = rfft(WIN .* @view(y[(i-1)*HOP .+ (1:FRAME_LEN), m]))
    end
    return S
end

function frontend(x, fs, osf)
    y, fsos = highpass_osf(x, fs, osf)
    S = stft3(y)
    freqs = (0:FRAME_LEN÷2) .* (fsos / FRAME_LEN)
    inband = findall(f -> F_MIN <= f <= F_MAX, freqs)
    return y, S, freqs, inband
end

function frame_energy_db(y)
    nf = 1 + div(size(y, 1) - FRAME_LEN, HOP)
    e = [sum(abs2, @view y[(i-1)*HOP .+ (1:FRAME_LEN), :]) for i in 1:nf]
    return 10 .* log10.(e ./ maximum(e) .+ 1e-30)
end

# ----------------------------------------------------------------------------
# 3. Your three algorithms, general-geometry versions
# ----------------------------------------------------------------------------

# --- MUSIC (script 1): energy-selected bins, 1%-eigenvalue source count,
#     energy-weighted fusion of normalised pseudo-spectra; returns the global peak.
function music_doa(x, fs, mics; osf = OSF.music, step = MUSIC_STEP_DEG)
    y, S, freqs, inband = frontend(x, fs, osf)
    nb, nf, _ = size(S)
    energy = [sum(abs2, @view S[b, :, :]) for b in 1:nb]; energy[1] = 0.0
    isempty(inband) && return NaN
    thr = 0.3 * maximum(energy[inband])
    sig = [j for j in inband if energy[j] > thr]
    angles = 0.0:step:360.0-step; U = unit(angles)
    t21 = tau1k(mics, 2, U); t31 = tau1k(mics, 3, U)
    fused = zeros(length(angles)); nused = 0
    for b in sig
        f = freqs[b]
        X = permutedims(S[b, :, :])                       # 3 x nf
        E = eigen(Hermitian((X * X') ./ nf))               # ascending eigenvalues
        ev = max.(E.values, eps())
        K = count(>(0.01 * maximum(ev)), ev)
        K >= 3 && continue
        En = E.vectors[:, 1:3-K]
        A = vcat(ones(1, length(angles)),
                 reshape(cis.(-2π * f .* t21), 1, :),
                 reshape(cis.(-2π * f .* t31), 1, :))
        spec = 1 ./ (vec(sum(abs2, En' * A, dims = 1)) .+ 1e-30)
        fused .+= energy[b] .* spec ./ maximum(spec)
        nused += 1
    end
    nused == 0 && return NaN
    return angles[argmax(fused)]
end

# --- Trig+FFT (script 2): strongest in-band bin of mic 1 per frame, phase -> (cos,sin) -> theta
function trig_doa(x, fs, mics; osf = OSF.trig)
    y, S, freqs, inband = frontend(x, fs, osf)
    nf = size(S, 2)
    D = vcat((mics[1, :] .- mics[2, :])', (mics[1, :] .- mics[3, :])')   # 2 x 2
    n12, n13 = norm(D[1, :]), norm(D[2, :])
    est = Vector{Float64}(undef, nf)
    for i in 1:nf
        b = inband[argmax(abs2.(S[inband, i, 1]))]
        λ = C_SOUND / freqs[b]
        ph12 = angle(S[b, i, 2] * conj(S[b, i, 1]))
        ph13 = angle(S[b, i, 3] * conj(S[b, i, 1]))
        s  = clamp((-ph12 * λ) / (2π * n12), -1, 1)
        cc = clamp((-ph13 * λ) / (2π * n13), -1, 1)
        u = D \ [s * n12, cc * n13]
        est[i] = mod(atand(u[2], u[1]), 360.0)
    end
    return (doa = circmean_deg(est), frames = est, edb = frame_energy_db(y))
end

# --- LUT + cost (script 3): wrapped-phase squared residual on a theta grid
function lut_doa(x, fs, mics; osf = OSF.lut, freq_mode = LUT_FREQ_MODE, step = LUT_STEP_DEG)
    y, S, freqs, inband = frontend(x, fs, osf)
    nf = size(S, 2)
    angles = 0.0:step:360.0-step; U = unit(angles)
    t21 = tau1k(mics, 2, U); t31 = tau1k(mics, 3, U)
    f_lut = freqs[inband[argmax(abs2.(S[inband, 1, 1]))]]            # first frame's tone
    est = Vector{Float64}(undef, nf)
    for i in 1:nf
        b = inband[argmax(abs2.(S[inband, i, 1]))]
        f = freq_mode == :first_frame ? f_lut : freqs[b]
        ph12 = angle(S[b, i, 2] * conj(S[b, i, 1]))
        ph13 = angle(S[b, i, 3] * conj(S[b, i, 1]))
        cost = wrappi.(-2π * f .* t21 .- ph12) .^ 2 .+ wrappi.(-2π * f .* t31 .- ph13) .^ 2
        est[i] = angles[argmin(cost)]
    end
    return (doa = circmean_deg(est), frames = est, edb = frame_energy_db(y))
end

# name => f(x, fs, mics) -> azimuth (deg)
const ALGOS = Pair{String,Function}[
    "MUSIC"    => (x, fs, m) -> music_doa(x, fs, m),
    "Trig+FFT" => (x, fs, m) -> trig_doa(x, fs, m).doa,
    "LUT+cost" => (x, fs, m) -> lut_doa(x, fs, m).doa,
]

# ----------------------------------------------------------------------------
# 4. Data + perturbation tools
# ----------------------------------------------------------------------------
function pick_segment(x, len)
    N = size(x, 1); len = min(len, N)
    cs = [0.0; cumsum(vec(sum(abs2, x, dims = 2)))]
    s = argmax([cs[i+len] - cs[i] for i in 1:(N-len+1)])
    return x[s:s+len-1, :]
end

"Build the recordings Dict from filenames like 180cm_270_d_man_sin_4000hz.wav (true angle = 2nd field)."
function discover_recordings(dir = REC_DIR; pattern = r"^\d+cm_[\d.]+_.*\.wav$")
    d = Dict{String,Float64}()
    for f in readdir(dir)
        occursin(pattern, f) && (d[f] = parse(Float64, split(f, '_')[2]))
    end
    return d
end

function load_all(recordings = RECORDINGS)
    recs = Dict{String,NamedTuple}()
    for (name, truth) in recordings
        raw, fs = WAV.wavread(joinpath(REC_DIR, name), format = "native")
        x = Float64.(raw[:, 1:3]); x .-= mean(x, dims = 1); fs = Float64(fs)
        seg = ANALYSIS_SECONDS === nothing ? x : pick_segment(x, round(Int, ANALYSIS_SECONDS * fs))
        recs[name] = (x = seg, full = x, fs = fs, truth = truth)
    end
    return recs
end

"Mean power of x inside F_MIN..min(F_MAX, fs/2), averaged over channels."
function band_power(x, fs)
    N = size(x, 1); f = rfftfreq(N, fs)
    mask = (f .>= F_MIN) .& (f .<= F_MAX)
    xb = irfft(rfft(x, 1) .* mask, N, 1)
    return mean(abs2, xb)
end

"Add white Gaussian noise so that ADDED-noise power in the analysis band = signal band power / 10^(snr/10)."
function add_noise_band(x, fs, snr_db, rng)
    fhi = min(F_MAX, fs / 2)
    σ2 = band_power(x, fs) / 10^(snr_db / 10) * (fs / 2) / (fhi - F_MIN)
    return x .+ sqrt(σ2) .* randn(rng, size(x))
end

"y_m(t) = g_m x_m(t - d_m); fractional delay by FFT phase ramp (d in samples)."
function apply_mismatch(x, gains_lin, delays_samp)
    N, M = size(x)
    X = rfft(x, 1); f = rfftfreq(N)
    for m in 1:M
        X[:, m] .*= gains_lin[m] .* cis.(-2π .* f .* delays_samp[m])
    end
    return irfft(X, N, 1)
end

# ----------------------------------------------------------------------------
# 5. Experiments  -> long DataFrames: exp, rec, algo, level, trial, est, err, delta
#    err = vs ground truth ; delta = vs the algorithm's clean (unperturbed) estimate
# ----------------------------------------------------------------------------
mkrow(e, rec, algo, level, trial, est, truth, b) =
    (exp = e, rec = rec, algo = algo, level = Float64(level), trial = trial, est = est,
     err = wrap180(est - truth), delta = wrap180(est - b))

recnames(recs) = sort(collect(keys(recs)))

baseline(recs) = Dict((rn, an) => f(recs[rn].x, recs[rn].fs, MICS_NOMINAL)
                      for rn in keys(recs) for (an, f) in ALGOS)

function experiment_A(recs, base; snrs = SNR_DB, ntrials = N_TRIALS, rng = MersenneTwister(SEED))
    rows = NamedTuple[]
    for rn in recnames(recs), snr in snrs, t in 1:ntrials
        r = recs[rn]; xn = add_noise_band(r.x, r.fs, snr, rng)
        for (an, f) in ALGOS
            push!(rows, mkrow("A", rn, an, snr, t, f(xn, r.fs, MICS_NOMINAL), r.truth, base[(rn, an)]))
        end
    end
    return DataFrame(rows)
end

# B: recordings untouched; the ASSUMED coordinates of all three mics are perturbed (Gaussian, per coordinate)
function experiment_B(recs, base; sigmas_mm = POS_SIGMA_MM, ntrials = N_TRIALS, rng = MersenneTwister(SEED + 1))
    rows = NamedTuple[]
    for σ in sigmas_mm, t in 1:ntrials
        mics = MICS_NOMINAL .+ (σ * 1e-3) .* randn(rng, 3, 2)
        for rn in recnames(recs)
            r = recs[rn]
            for (an, f) in ALGOS
                push!(rows, mkrow("B", rn, an, σ, t, f(r.x, r.fs, mics), r.truth, base[(rn, an)]))
            end
        end
    end
    return DataFrame(rows)
end

# C: per-channel gain (dB) and timing (us) errors, applied to the recording; assumed geometry nominal.
# bias_* = measured systematic offsets per channel, added on top of the random part.
function experiment_C(recs, base; gain_sigmas_db = GAIN_SIGMA_DB, delay_sigmas_us = DELAY_SIGMA_US,
                      ntrials = N_TRIALS, rng = MersenneTwister(SEED + 2),
                      bias_gain_db = zeros(3), bias_delay_us = zeros(3))
    rows = NamedTuple[]
    for (factor, sigmas) in (("C-gain", gain_sigmas_db), ("C-delay", delay_sigmas_us)), σ in sigmas, t in 1:ntrials
        g_db = bias_gain_db  .+ (factor == "C-gain"  ? σ .* randn(rng, 3) : zeros(3))
        d_us = bias_delay_us .+ (factor == "C-delay" ? σ .* randn(rng, 3) : zeros(3))
        for rn in recnames(recs)
            r = recs[rn]
            xm = apply_mismatch(r.x, 10 .^ (g_db ./ 20), d_us .* 1e-6 .* r.fs)
            for (an, f) in ALGOS
                push!(rows, mkrow(factor, rn, an, σ, t, f(xm, r.fs, MICS_NOMINAL), r.truth, base[(rn, an)]))
            end
        end
    end
    return DataFrame(rows)
end

# D1: the per-STFT-frame estimates your Trig and LUT scripts average (full recording).
function experiment_D_frames(recs)
    rows = NamedTuple[]
    for rn in recnames(recs)
        r = recs[rn]
        for (an, fn) in ("Trig+FFT" => trig_doa, "LUT+cost" => lut_doa)
            o = fn(r.full, r.fs, MICS_NOMINAL)
            for i in eachindex(o.frames)
                push!(rows, (rec = rn, algo = an, frame = i, edb = o.edb[i], est = o.frames[i],
                             truth = r.truth, err = wrap180(o.frames[i] - r.truth)))
            end
        end
    end
    return DataFrame(rows)
end

"Per-frame spread and what the circular-mean fusion gives, for several energy gates."
function gate_summary(d1; gates = [Inf, 30.0, 20.0, 10.0])
    rows = NamedTuple[]
    for rn in unique(d1.rec), an in unique(d1.algo), g in gates
        sub = d1[(d1.rec .== rn) .& (d1.algo .== an) .& (d1.edb .>= -g), :]
        isempty(sub) && continue
        fused = circmean_deg(sub.est); st = stats(sub.err)
        push!(rows, (rec = rn, algo = an, gate_db = g, nframes = nrow(sub), fused_deg = fused,
                     fused_err = wrap180(fused - first(sub.truth)), frame_sd = st.sd,
                     frame_p95 = st.p95, frame_outlier_pct = st.outlier_pct))
    end
    return DataFrame(rows)
end

# D2: Monte Carlo over random (energetic) observation windows -- all three algorithms, full pipeline
function experiment_D_blocks(recs, base; lens = BLOCK_LENS, nblocks = N_BLOCKS, vad_db = VAD_DB,
                             rng = MersenneTwister(SEED + 3))
    rows = NamedTuple[]
    for rn in recnames(recs), L in lens
        r = recs[rn]; x = r.full; N = size(x, 1)
        N < L && continue
        cs = [0.0; cumsum(vec(sum(abs2, x, dims = 2)))]
        starts = collect(1:HOP:(N-L+1))
        e = [cs[s+L] - cs[s] for s in starts]
        cands = starts[10 .* log10.(e ./ maximum(e) .+ 1e-30) .>= -vad_db]
        for t in 1:nblocks
            s = rand(rng, cands); blk = x[s:s+L-1, :]
            for (an, f) in ALGOS
                push!(rows, mkrow("D-block", rn, an, L, t, f(blk, r.fs, MICS_NOMINAL), r.truth, base[(rn, an)]))
            end
        end
    end
    return DataFrame(rows)
end

"Diagnostic: channel RMS ratios (dB) over the recording, relative to mic 1."
channel_gain_db(x) = 20 .* log10.(vec(sqrt.(mean(abs2, x, dims = 1))) ./ sqrt(mean(abs2, x[:, 1])))

# ----------------------------------------------------------------------------
# 6. Statistics, budget, plots
# ----------------------------------------------------------------------------
function stats(v)
    nan_pct = 100 * mean(isnan.(v))
    v = filter(isfinite, v)
    isempty(v) && return (n = 0, nan_pct = nan_pct, bias = NaN, sd = NaN, rmse = NaN, mae = NaN, p95 = NaN, outlier_pct = NaN)
    return (n = length(v), nan_pct = nan_pct, bias = mean(v), sd = std(v), rmse = sqrt(mean(abs2, v)),
            mae = mean(abs, v), p95 = quantile(abs.(v), 0.95), outlier_pct = 100 * mean(abs.(v) .> OUTLIER_DEG))
end

"col=:err -> accuracy vs truth ; col=:delta -> sensitivity (shift from the clean estimate)."
summarize(df; col = :err) = combine(groupby(df, [:exp, :algo, :level]), col => (v -> stats(v)) => AsTable)

function error_budget(S)
    sub = subset(S, [:exp, :level] => ByRow((e, l) -> get(TYPICAL, e, nothing) == l))
    return sort(select(sub, :exp, :level, :algo, :sd, :rmse, :p95, :outlier_pct), [:exp, :algo])
end

"Print a whole table without Julia's row/column truncation."
showall(df) = show(stdout, MIME("text/plain"), df; allrows = true, allcols = true, summary = false)

"Excel-safe copy: NaN/Inf -> missing."
xl(df) = mapcols(c -> [x isa AbstractFloat && !isfinite(x) ? missing : x for x in c], df)

function plot_sweep(S, e; metric = :rmse, xlabel = "level", xflip = false, logx = false)
    sub = S[S.exp .== e, :]
    p = plot(; xlabel = xlabel, ylabel = "$(metric) (deg)", title = "Experiment $e", legend = :topleft,
             xflip = xflip, xscale = logx ? :log10 : :identity)
    for g in groupby(sub, :algo)
        sg = sort(g, :level); plot!(p, sg.level, sg[!, metric]; marker = :o, label = first(g.algo))
    end
    return p
end

function plot_frame_hist(d1; gate_db = Inf, bins = -60:2:60)
    sub = d1[d1.edb .>= -gate_db, :]
    p = plot(; xlabel = "per-frame error (deg)", ylabel = "frames", title = "D1 frame errors (gate $(gate_db) dB)")
    for g in groupby(sub, :algo)
        histogram!(p, g.err; bins = bins, alpha = 0.5, label = first(g.algo))
    end
    return p
end

# ----------------------------------------------------------------------------
# 7. Driver
# ----------------------------------------------------------------------------
function run_experiments(; outdir = "results", ntrials = N_TRIALS, nblocks = N_BLOCKS, recordings = RECORDINGS)
    mkpath(outdir)
    recs = load_all(recordings)      # e.g. recordings = discover_recordings() for every matching file
    base = baseline(recs)
    clean = DataFrame([(rec = rn, algo = an, clean_est_deg = v, truth_deg = recs[rn].truth,
                        err_deg = wrap180(v - recs[rn].truth)) for ((rn, an), v) in sort(collect(base))])
    CSV.write(joinpath(outdir, "clean_estimates.csv"), clean)
    gains = DataFrame([(rec = rn, mic1_db = 0.0, mic2_db = g[2], mic3_db = g[3])
                       for rn in recnames(recs) for g in (channel_gain_db(recs[rn].full),)])
    CSV.write(joinpath(outdir, "channel_gains.csv"), gains)

    dfA = experiment_A(recs, base; ntrials = ntrials)
    dfB = experiment_B(recs, base; ntrials = ntrials)
    dfC = experiment_C(recs, base; ntrials = ntrials)
    dfD2 = experiment_D_blocks(recs, base; nblocks = nblocks)
    dfD1 = experiment_D_frames(recs)
    dfs = Dict("A" => dfA, "B" => dfB, "C" => dfC, "D2" => dfD2)

    for (k, df) in dfs
        CSV.write(joinpath(outdir, "exp$(k)_raw.csv"), df)
        CSV.write(joinpath(outdir, "exp$(k)_summary_err.csv"),   summarize(df; col = :err))
        CSV.write(joinpath(outdir, "exp$(k)_summary_delta.csv"), summarize(df; col = :delta))
    end
    CSV.write(joinpath(outdir, "expD1_frames_raw.csv"), dfD1)
    G = gate_summary(dfD1); CSV.write(joinpath(outdir, "expD1_gate_summary.csv"), G)

    Sd = vcat((summarize(df; col = :delta) for df in values(dfs))...)
    budget = error_budget(Sd); CSV.write(joinpath(outdir, "error_budget.csv"), budget)

    # One Excel workbook with every summary table (raw per-trial data stays in the CSVs: too big for sheets)
    Gx = transform(G, :gate_db => ByRow(g -> isinf(g) ? "all" : string(g)) => :gate_db)
    sheets = Pair{String,DataFrame}["error_budget" => xl(budget), "clean_estimates" => xl(clean),
                                    "D1_gate_summary" => xl(Gx)]
    for k in sort(collect(keys(dfs)))
        push!(sheets, "$(k)_delta" => xl(summarize(dfs[k]; col = :delta)),
                      "$(k)_err"   => xl(summarize(dfs[k]; col = :err)))
    end
    try
        XLSX.writetable(joinpath(outdir, "summary.xlsx"), sheets...; overwrite = true)
    catch err
        @warn "Excel export skipped (CSVs are already saved)" err
    end

    savefig(plot_sweep(Sd, "A"; xlabel = "added-noise in-band SNR (dB)", xflip = true), joinpath(outdir, "A_vs_snr.png"))
    savefig(plot_sweep(Sd, "B"; xlabel = "mic position σ (mm)"),                         joinpath(outdir, "B_vs_position.png"))
    savefig(plot_sweep(Sd, "C-gain"; xlabel = "gain σ (dB)"),                            joinpath(outdir, "C_gain.png"))
    savefig(plot_sweep(Sd, "C-delay"; xlabel = "timing σ (µs)"),                         joinpath(outdir, "C_delay.png"))
    savefig(plot_sweep(Sd, "D-block"; xlabel = "observation length (samples)", logx = true), joinpath(outdir, "D2_vs_blocklen.png"))
    savefig(plot_frame_hist(dfD1; gate_db = Inf),                                        joinpath(outdir, "D1_frame_hist_all.png"))
    savefig(plot_frame_hist(dfD1; gate_db = 20.0),                                       joinpath(outdir, "D1_frame_hist_gate20.png"))
    println("Done. Results saved to: ", abspath(outdir))
    return nothing          # nothing returned, so Jupyter prints no table
end

run_experiments (generic function with 1 method)

In [20]:
run_experiments(ntrials = 20)

Done. Results saved to: C:\Users\Morena Mofokeng\Desktop\Research project\results
